# Stage 3: Semantic Chunking Engine Test Notebook

This notebook tests **Stage 3: The Semantic Chunking Engine**, which converts **raw extracted content** from OCR tools into Logical Document Units (LDUs) that are RAG-ready and preserve structural context.

## Complete Pipeline Flow

```
PDF Document
    ↓
Stage 1: Triage Agent → DocumentProfile
    ↓
Stage 2: Extraction Router → OCR Tools (Docling/MinerU/VisionExtractor)
    ↓
    Raw OCR Output → Normalized to ExtractedDocument
    ↓
    Save to Output Folder: {doc_id}_extracted.json & {doc_id}_extracted.md
    ↓
Stage 3: Semantic Chunking Engine ← Load from Output Folder (parsed content)
    ↓
    List[LDU] (RAG-ready semantic chunks)
```

**Key Point**: The ChunkingEngine loads the **parsed content** (markdown/JSON) from the output folder (`outputs/`) and applies semantic chunking rules to create LDUs.

## Testing Focus

1. **Raw OCR Output → ExtractedDocument** - Verify OCR tools output normalized ExtractedDocument
2. **ExtractedDocument → LDUs** - Verify raw extracted content is passed to chunking engine
3. **LDU Structure** - Verify each LDU has all required fields: content, chunk_type, page_refs, bounding_box, parent_section, token_count, content_hash
4. **Chunking Rules Validation** - Test all 5 core chunking rules:
   - **Rule 1**: Table cell never split from header row
   - **Rule 2**: Figure caption stored as metadata of parent figure chunk
   - **Rule 3**: Numbered list kept as single LDU (unless exceeds max_tokens)
   - **Rule 4**: Section headers stored as parent metadata on child chunks
   - **Rule 5**: Cross-references resolved and stored as chunk relationships
5. **Markdown Export** - Save raw extracted content and chunks as markdown for inspection

## Setup

In [1]:
import json
from pathlib import Path
from typing import Dict, List, Optional

# Project imports
from src.agents.chunker import ChunkingEngine
from src.agents.extractor import ExtractionRouter
from src.agents.triage import TriageAgent
from src.models.extracted_document import ExtractedDocument
from src.models.ldu import LDU

# Setup paths
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"  # User's output directory
PROFILES_DIR = BASE_DIR / ".refinery" / "profiles"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROFILES_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Imports successful")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Profiles directory: {PROFILES_DIR}")

c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Imports successful
✓ Output directory: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs
✓ Profiles directory: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\.refinery\profiles


## Step 1: Select Test Document

Select a PDF document from the `data/` directory to test semantic chunking.

In [2]:
# Select a test document (modify as needed)
# Example: Class B - Scanned Document
pdf_path = DATA_DIR / "class_b" / "2018_Audited_Financial_Statement_Report.pdf"

# Validate PDF exists
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

print(f"✓ Selected document: {pdf_path.name}")
print(f"  Path: {pdf_path}")
print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")

doc_id_base = pdf_path.stem  # Use filename without extension as base
print(f"  Document ID base: {doc_id_base}")

✓ Selected document: 2018_Audited_Financial_Statement_Report.pdf
  Path: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_b\2018_Audited_Financial_Statement_Report.pdf
  Size: 1.08 MB
  Document ID base: 2018_Audited_Financial_Statement_Report


## Step 2: Check for Existing Parsed Content

Check if parsed content already exists in the output folder. If found, we can skip extraction and proceed directly to chunking.

In [3]:
# Check if parsed content already exists in output folder
# Look for files in various locations:
# 1. Flat structure: outputs/{doc_id}_extracted.json or .md
# 2. Docling structure: outputs/docling/markdown/{doc_id}_*/{doc_name}.md
# 3. MinerU structure: outputs/mineru/{doc_name}.md or .json

# Search for existing parsed content
existing_json = None
existing_md = None
found_source = None  # Track which source we found

# Check flat structure first
flat_json = OUTPUT_DIR / f"{doc_id_base}_extracted.json"
flat_md = OUTPUT_DIR / f"{doc_id_base}_extracted.md"
if flat_json.exists():
    existing_json = flat_json
    found_source = "flat_json"
elif flat_md.exists():
    existing_md = flat_md
    found_source = "flat_md"

# Check Docling structure: outputs/docling/markdown/{doc_id}_*/{doc_name}.md
if not existing_json and not existing_md:
    docling_md_dir = OUTPUT_DIR / "docling" / "markdown"
    if docling_md_dir.exists():
        # Search for directories matching the doc_id pattern
        for subdir in docling_md_dir.glob(f"{doc_id_base}*"):
            md_file = subdir / f"{doc_id_base}.md"
            if md_file.exists():
                existing_md = md_file
                found_source = "docling"
                print(f"📁 Found Docling markdown: {md_file}")
                break

# Check MinerU structure
if not existing_json and not existing_md:
    mineru_dir = OUTPUT_DIR / "mineru"
    if mineru_dir.exists():
        # Check for nested structure: outputs/mineru/{doc_name}/hybrid_auto/{doc_name}.md
        mineru_subdir = mineru_dir / doc_id_base
        if mineru_subdir.exists():
            # Check hybrid_auto subdirectory (common MinerU output location)
            hybrid_auto_dir = mineru_subdir / "hybrid_auto"
            if hybrid_auto_dir.exists():
                mineru_json = hybrid_auto_dir / f"{doc_id_base}_content_list_v2.json"
                mineru_md = hybrid_auto_dir / f"{doc_id_base}.md"
                if mineru_json.exists():
                    existing_json = mineru_json
                    found_source = "mineru_json"
                elif mineru_md.exists():
                    existing_md = mineru_md
                    found_source = "mineru_md"
            # Also check directly in subdir (fallback)
            if not existing_json and not existing_md:
                mineru_json = mineru_subdir / f"{doc_id_base}_content_list_v2.json"
                mineru_md = mineru_subdir / f"{doc_id_base}.md"
                if mineru_json.exists():
                    existing_json = mineru_json
                    found_source = "mineru_json"
                elif mineru_md.exists():
                    existing_md = mineru_md
                    found_source = "mineru_md"
        # Check flat structure
        if not existing_json and not existing_md:
            mineru_json = mineru_dir / f"{doc_id_base}.json"
            mineru_md = mineru_dir / f"{doc_id_base}.md"
            if mineru_json.exists():
                existing_json = mineru_json
                found_source = "mineru_json"
            elif mineru_md.exists():
                existing_md = mineru_md
                found_source = "mineru_md"

if existing_json or existing_md:
    print("📁 Found existing parsed content in output folder!")
    found_file = existing_json if existing_json else existing_md
    print(f"  → Source: {found_source}")
    print(f"  → File: {found_file}")
    print("  → Extraction steps (3A/3B) will be skipped")
    print("  → Step 4 will use this existing content for chunking")
    print("  → To re-extract, set HAS_EXISTING_CONTENT = False below\n")
    HAS_EXISTING_CONTENT = True
    # Store which source was found for Step 4
    PREFERRED_SOURCE = found_source
else:
    HAS_EXISTING_CONTENT = False
    PREFERRED_SOURCE = None
    print("📄 No existing parsed content found.")
    print("  → You can run Step 3A (Docling) or Step 3B (MinerU) to extract content\n")

📁 Found existing parsed content in output folder!
  → Source: flat_json
  → File: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_extracted.json
  → Extraction steps (3A/3B) will be skipped
  → Step 4 will use this existing content for chunking
  → To re-extract, set HAS_EXISTING_CONTENT = False below



## Step 3A: Extract with Docling (Optional)

**Only run this if you don't have existing parsed content or want to re-extract with Docling.**

This step:
1. Runs Triage Agent to classify the document
2. Uses Docling (LayoutExtractor) to extract structured content
3. Normalizes output to ExtractedDocument schema
4. Saves to output folder as JSON and Markdown

In [4]:
# Step 3A: Extract with Docling
# Only run this if you don't have existing content or want to re-extract

if HAS_EXISTING_CONTENT:
    print("⏭️  Skipping Docling extraction - existing parsed content found")
    print("  → If you want to re-extract, set HAS_EXISTING_CONTENT = False above\n")
else:
    print("📄 Step 3A: Extracting with Docling (LayoutExtractor)...\n")
    
    # Initialize triage agent and extraction router
    triage_agent = TriageAgent(profiles_dir=PROFILES_DIR)
    extraction_router = ExtractionRouter()
    
    # Classify document
    print("🔍 Running triage analysis...")
    profile = triage_agent.classify_document(pdf_path)
    
    print(f"\n📊 Document Profile:")
    print(f"  Document ID: {profile.doc_id}")
    print(f"  Origin Type: {profile.origin_type}")
    print(f"  Layout Complexity: {profile.layout_complexity}")
    print(f"  Domain Hint: {profile.domain_hint}")
    print(f"  Estimated Cost: {profile.estimated_cost}")
    print(f"  Page Count: {profile.metadata.page_count}")
    
    # Extract document using Docling (LayoutExtractor)
    print("\n📄 Extracting with Docling...")
    print("  → This may take a few minutes for large documents...")
    
    try:
        extracted_doc = extraction_router.extract(profile, str(pdf_path))
        doc_id = profile.doc_id
        
        print(f"\n✓ Extraction Complete:")
        print(f"  Text Blocks: {len(extracted_doc.text_blocks)}")
        print(f"  Tables: {len(extracted_doc.tables)}")
        print(f"  Figures: {len(extracted_doc.figures)}")
        
        # Save raw extracted document as JSON
        extracted_json_path = OUTPUT_DIR / f"{doc_id}_extracted.json"
        extracted_json_path.write_text(extracted_doc.model_dump_json(indent=2), encoding="utf-8")
        print(f"\n✓ Saved ExtractedDocument JSON: {extracted_json_path}")
        
        # Convert to markdown
        def extracted_doc_to_markdown(doc: ExtractedDocument, title: str = "Extracted Document") -> str:
            """Convert ExtractedDocument to Markdown format."""
            lines = [f"# {title}\n"]
            lines.append(f"**Extracted from:** {pdf_path.name}\n")
            lines.append(f"**Text Blocks:** {len(doc.text_blocks)}, **Tables:** {len(doc.tables)}, **Figures:** {len(doc.figures)}\n")
            
            # Text blocks
            if doc.text_blocks:
                lines.append("## Text Blocks\n")
                for i, block in enumerate(doc.text_blocks, 1):
                    lines.append(f"### Text Block {i} (Page {block.page_num})")
                    lines.append(f"**Bounding Box:** ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")
                    lines.append(f"\n{block.content}\n")
            
            # Tables
            if doc.tables:
                lines.append("## Tables\n")
                for i, table in enumerate(doc.tables, 1):
                    lines.append(f"### Table {i} (Page {table.page_num})")
                    lines.append(f"**Bounding Box:** ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")
                    lines.append("\n")
                    if table.headers:
                        lines.append("| " + " | ".join(table.headers) + " |")
                        lines.append("| " + " | ".join(["---"] * len(table.headers)) + " |")
                    for row in table.rows:
                        lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
                    lines.append("\n")
            
            # Figures
            if doc.figures:
                lines.append("## Figures\n")
                for i, figure in enumerate(doc.figures, 1):
                    lines.append(f"### Figure {i} (Page {figure.page_num})")
                    lines.append(f"**Bounding Box:** ({figure.bbox.x0:.1f}, {figure.bbox.y0:.1f}) → ({figure.bbox.x1:.1f}, {figure.bbox.y1:.1f})")
                    if figure.caption:
                        lines.append(f"**Caption:** {figure.caption}")
                    lines.append("\n")
            
            return "\n".join(lines)
        
        # Save markdown
        extracted_md = extracted_doc_to_markdown(extracted_doc, f"Raw Extracted Content: {pdf_path.name}")
        extracted_md_path = OUTPUT_DIR / f"{doc_id}_extracted.md"
        extracted_md_path.write_text(extracted_md, encoding="utf-8")
        print(f"✓ Saved Markdown: {extracted_md_path}")
        print(f"\n  → These files will be used by semantic chunking stage")
        
    except Exception as e:
        print(f"\n❌ Docling extraction failed: {e}")
        print("  → You can try Step 3B (MinerU) instead, or use existing parsed content")

⏭️  Skipping Docling extraction - existing parsed content found
  → If you want to re-extract, set HAS_EXISTING_CONTENT = False above



## Step 3B: Extract with MinerU (Optional)

**Only run this if you don't have existing parsed content or want to re-extract with MinerU.**

This step:
1. Runs Triage Agent to classify the document
2. Uses MinerU to extract structured content
3. Normalizes output to ExtractedDocument schema
4. Saves to output folder as JSON and Markdown

**Note**: MinerU requires the JSON file to be pre-generated. If not found, you'll need to run MinerU CLI first.

In [5]:
# Step 3B: Extract with MinerU
# Only run this if you don't have existing content or want to re-extract

if HAS_EXISTING_CONTENT:
    print("⏭️  Skipping MinerU extraction - existing parsed content found")
    print("  → If you want to re-extract, set HAS_EXISTING_CONTENT = False above\n")
else:
    print("📄 Step 3B: Extracting with MinerU...\n")
    
    # Check for MinerU JSON file
    mineru_json_path = pdf_path.with_suffix('.pdf.mineru.json')
    
    # Also check in outputs/mineru directory
    mineru_dir = OUTPUT_DIR / "mineru" / doc_id_base
    if mineru_dir.exists():
        mineru_json_path = mineru_dir / f"{doc_id_base}_content_list_v2.json"
        if not mineru_json_path.exists():
            mineru_json_path = mineru_dir / f"{doc_id_base}_content_list.json"
    
    if not mineru_json_path.exists():
        print(f"⚠️  MinerU JSON not found at: {mineru_json_path}")
        print("\n📝 To generate MinerU JSON, run one of these commands:")
        print(f"  Option 1 (CLI): mineru extract \"{pdf_path}\" -o \"{OUTPUT_DIR / 'mineru'}\" --format json")
        print(f"  Option 2 (Python): Use scripts/generate_mineru_json.py")
        print(f"\n  Then ensure the output JSON is saved to: {mineru_json_path}\n")
        print("⏭️  Skipping MinerU extraction - JSON file required")
    else:
        print(f"✓ Found MinerU JSON: {mineru_json_path}\n")
        
        # Initialize triage agent and extraction router
        triage_agent = TriageAgent(profiles_dir=PROFILES_DIR)
        extraction_router = ExtractionRouter()
        
        # Classify document
        print("🔍 Running triage analysis...")
        profile = triage_agent.classify_document(pdf_path)
        
        print(f"\n📊 Document Profile:")
        print(f"  Document ID: {profile.doc_id}")
        print(f"  Origin Type: {profile.origin_type}")
        print(f"  Layout Complexity: {profile.layout_complexity}")
        print(f"  Domain Hint: {profile.domain_hint}")
        print(f"  Estimated Cost: {profile.estimated_cost}")
        print(f"  Page Count: {profile.metadata.page_count}")
        
        # Extract document using MinerU
        print("\n📄 Extracting with MinerU...")
        
        try:
            from src.strategies.layout_mineru import MinerUExtractor
            mineru_extractor = MinerUExtractor(mineru_json_path=str(mineru_json_path))
            extracted_doc = mineru_extractor.extract(str(pdf_path))
            doc_id = profile.doc_id
            
            print(f"\n✓ Extraction Complete:")
            print(f"  Text Blocks: {len(extracted_doc.text_blocks)}")
            print(f"  Tables: {len(extracted_doc.tables)}")
            print(f"  Figures: {len(extracted_doc.figures)}")
            
            # Save raw extracted document as JSON
            extracted_json_path = OUTPUT_DIR / f"{doc_id}_extracted.json"
            extracted_json_path.write_text(extracted_doc.model_dump_json(indent=2), encoding="utf-8")
            print(f"\n✓ Saved ExtractedDocument JSON: {extracted_json_path}")
            
            # Convert to markdown (same function as Docling)
            def extracted_doc_to_markdown(doc: ExtractedDocument, title: str = "Extracted Document") -> str:
                """Convert ExtractedDocument to Markdown format."""
                lines = [f"# {title}\n"]
                lines.append(f"**Extracted from:** {pdf_path.name}\n")
                lines.append(f"**Text Blocks:** {len(doc.text_blocks)}, **Tables:** {len(doc.tables)}, **Figures:** {len(doc.figures)}\n")
                
                # Text blocks
                if doc.text_blocks:
                    lines.append("## Text Blocks\n")
                    for i, block in enumerate(doc.text_blocks, 1):
                        lines.append(f"### Text Block {i} (Page {block.page_num})")
                        lines.append(f"**Bounding Box:** ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")
                        lines.append(f"\n{block.content}\n")
                
                # Tables
                if doc.tables:
                    lines.append("## Tables\n")
                    for i, table in enumerate(doc.tables, 1):
                        lines.append(f"### Table {i} (Page {table.page_num})")
                        lines.append(f"**Bounding Box:** ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")
                        lines.append("\n")
                        if table.headers:
                            lines.append("| " + " | ".join(table.headers) + " |")
                            lines.append("| " + " | ".join(["---"] * len(table.headers)) + " |")
                        for row in table.rows:
                            lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
                        lines.append("\n")
                
                # Figures
                if doc.figures:
                    lines.append("## Figures\n")
                    for i, figure in enumerate(doc.figures, 1):
                        lines.append(f"### Figure {i} (Page {figure.page_num})")
                        lines.append(f"**Bounding Box:** ({figure.bbox.x0:.1f}, {figure.bbox.y0:.1f}) → ({figure.bbox.x1:.1f}, {figure.bbox.y1:.1f})")
                        if figure.caption:
                            lines.append(f"**Caption:** {figure.caption}")
                        lines.append("\n")
                
                return "\n".join(lines)
            
            # Save markdown
            extracted_md = extracted_doc_to_markdown(extracted_doc, f"Raw Extracted Content: {pdf_path.name}")
            extracted_md_path = OUTPUT_DIR / f"{doc_id}_extracted.md"
            extracted_md_path.write_text(extracted_md, encoding="utf-8")
            print(f"✓ Saved Markdown: {extracted_md_path}")
            print(f"\n  → These files will be used by semantic chunking stage")
            
        except Exception as e:
            print(f"\n❌ MinerU extraction failed: {e}")
            print("  → Check that MinerU JSON file is valid and in the correct format")

⏭️  Skipping MinerU extraction - existing parsed content found
  → If you want to re-extract, set HAS_EXISTING_CONTENT = False above



## Step 4: Semantic Chunking - Load Parsed Content and Create LDUs

**Pipeline Flow:**
1. **Load from Output Folder**: Read markdown/parsed content from `outputs/` directory
2. **ChunkingEngine**: Processes the parsed content and applies chunking rules
3. **Output**: `List[LDU]` (RAG-ready semantic chunks)

The ChunkingEngine loads the **parsed content** (markdown/JSON) from the output folder and converts it into semantically coherent LDUs.

In [6]:
# Step 4: Load parsed content and run semantic chunking

# Initialize chunking engine
chunking_engine = ChunkingEngine()

# Load parsed content from output folder
print("🧩 Step 4: Running Semantic Chunking Engine...")
print("  Loading parsed content from output folder...\n")

# Determine doc_id - use from extraction if available, otherwise from filename
if 'doc_id' not in locals():
    doc_id = doc_id_base  # Use filename without extension

# Search for existing parsed content (reuse logic from Step 2)
found_json = None
found_md = None

# Check flat structure
flat_json = OUTPUT_DIR / f"{doc_id}_extracted.json"
flat_md = OUTPUT_DIR / f"{doc_id}_extracted.md"
if flat_json.exists():
    found_json = flat_json
elif flat_md.exists():
    found_md = flat_md

# Check Docling structure
if not found_json and not found_md:
    docling_md_dir = OUTPUT_DIR / "docling" / "markdown"
    if docling_md_dir.exists():
        for subdir in docling_md_dir.glob(f"{doc_id}*"):
            md_file = subdir / f"{doc_id}.md"
            if md_file.exists():
                found_md = md_file
                break

# Check MinerU structure
if not found_json and not found_md:
    mineru_dir = OUTPUT_DIR / "mineru"
    if mineru_dir.exists():
        mineru_subdir = mineru_dir / doc_id
        if mineru_subdir.exists():
            # Check hybrid_auto subdirectory (common MinerU output location)
            hybrid_auto_dir = mineru_subdir / "hybrid_auto"
            if hybrid_auto_dir.exists():
                mineru_json = hybrid_auto_dir / f"{doc_id}_content_list_v2.json"
                mineru_md = hybrid_auto_dir / f"{doc_id}.md"
                if mineru_json.exists():
                    found_json = mineru_json
                elif mineru_md.exists():
                    found_md = mineru_md
            # Also check directly in subdir (fallback)
            if not found_json and not found_md:
                mineru_json = mineru_subdir / f"{doc_id}_content_list_v2.json"
                mineru_md = mineru_subdir / f"{doc_id}.md"
                if mineru_json.exists():
                    found_json = mineru_json
                elif mineru_md.exists():
                    found_md = mineru_md
        if not found_json and not found_md:
            mineru_json = mineru_dir / f"{doc_id}.json"
            mineru_md = mineru_dir / f"{doc_id}.md"
            if mineru_json.exists():
                found_json = mineru_json
            elif mineru_md.exists():
                found_md = mineru_md

# Load from found file (prefer JSON for full structure, but markdown works too)
if found_json:
    print(f"  → Loading from: {found_json}")
    # Load ExtractedDocument from JSON
    extracted_doc_data = json.loads(found_json.read_text(encoding="utf-8"))
    extracted_doc = ExtractedDocument(**extracted_doc_data)
    print(f"  ✓ Loaded ExtractedDocument from JSON")
    print(f"    Text Blocks: {len(extracted_doc.text_blocks)}")
    print(f"    Tables: {len(extracted_doc.tables)}")
    print(f"    Figures: {len(extracted_doc.figures)}")
elif found_md:
    print(f"  → Found markdown file: {found_md}")
    print(f"  → Converting markdown to ExtractedDocument...")
    print(f"  ⚠️  Note: Markdown parsing is simplified - JSON preserves full structure")
    
    # Parse markdown to create ExtractedDocument
    md_content = found_md.read_text(encoding="utf-8")
    
    from src.models.extracted_document import TextBlock, Table, Figure, BoundingBox
    
    text_blocks = []
    tables = []
    figures = []
    page_num = 1
    
    lines = md_content.split('\n')
    current_text_block = []
    current_table_rows = []
    in_table = False
    table_headers = None
    
    for i, line in enumerate(lines):
        stripped = line.strip()
        
        # Detect markdown tables (starts with | and contains |)
        if '|' in stripped and stripped.startswith('|'):
            # Skip table separator rows (|---|---|)
            if all(c in ['-', '|', ':', ' '] for c in stripped):
                continue
            
            if not in_table:
                in_table = True
                # Extract headers from first table row
                if table_headers is None:
                    headers = [cell.strip() for cell in stripped.split('|')[1:-1]]
                    table_headers = headers
                    current_table_rows = []
            else:
                # Data row
                cells = [cell.strip() for cell in stripped.split('|')[1:-1]]
                if len(cells) == len(table_headers) if table_headers else True:
                    current_table_rows.append(cells)
            continue
        elif in_table:
            # End of table - save it
            if current_table_rows and table_headers:
                tables.append(Table(
                    headers=table_headers,
                    rows=current_table_rows,
                    bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                    page_num=page_num
                ))
            in_table = False
            table_headers = None
            current_table_rows = []
        
        # Detect images/figures
        if stripped == '<!-- image -->':
            figures.append(Figure(
                caption="",
                bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                page_num=page_num
            ))
            continue
        
        # Regular text content
        if stripped and not stripped.startswith('#'):
            current_text_block.append(line)
        elif stripped.startswith('#'):
            # Header - save current block and start new one
            if current_text_block:
                text = '\n'.join(current_text_block).strip()
                if text:
                    text_blocks.append(TextBlock(
                        content=text,
                        bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                        page_num=page_num
                    ))
                current_text_block = []
    
    # Save last text block
    if current_text_block:
        text = '\n'.join(current_text_block).strip()
        if text:
            text_blocks.append(TextBlock(
                content=text,
                bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                page_num=page_num
            ))
    
    # Save last table if still in table
    if in_table and current_table_rows:
        tables.append(Table(
            headers=table_headers or [],
            rows=current_table_rows,
            bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
            page_num=page_num
        ))
    
    # Create ExtractedDocument from parsed markdown
    extracted_doc = ExtractedDocument(
        text_blocks=text_blocks,
        tables=tables,
        figures=figures,
        reading_order=list(range(len(text_blocks)))
    )
    print(f"  ✓ Created ExtractedDocument from markdown")
    print(f"    Text Blocks: {len(extracted_doc.text_blocks)}")
    print(f"    Tables: {len(extracted_doc.tables)}")
    print(f"    Figures: {len(extracted_doc.figures)}")
elif 'extracted_doc' in locals():
    print(f"  → Using in-memory ExtractedDocument from Step 3A/3B")
else:
    raise FileNotFoundError(
        f"No parsed content found in {OUTPUT_DIR}\n"
        f"Searched locations:\n"
        f"  - {OUTPUT_DIR}/{doc_id}_extracted.json\n"
        f"  - {OUTPUT_DIR}/{doc_id}_extracted.md\n"
        f"  - {OUTPUT_DIR}/docling/markdown/{doc_id}*/\n"
        f"  - {OUTPUT_DIR}/mineru/{doc_id}.*\n"
        f"Please run Step 3A (Docling) or Step 3B (MinerU) first to extract content."
    )

print(f"\n  Input: Parsed content from output folder")
print(f"  Processing: Applying chunking rules to parsed content\n")

🧩 Step 4: Running Semantic Chunking Engine...
  Loading parsed content from output folder...

  → Loading from: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_extracted.json
  ✓ Loaded ExtractedDocument from JSON
    Text Blocks: 1
    Tables: 3
    Figures: 8

  Input: Parsed content from output folder
  Processing: Applying chunking rules to parsed content



## Step 2: Extract Document & Save Parsed Content (Optional)

**Note**: This step is **optional** if parsed content already exists in the output folder. The semantic chunking stage (Step 3) will automatically load from the output folder.

**Pipeline Flow:**
1. **OCR Tools** (Docling/MinerU/VisionExtractor) extract raw content from PDF
2. **Normalization** → Raw OCR output is converted to normalized `ExtractedDocument` schema
3. **Save to Output Folder** → Convert ExtractedDocument to markdown/parsed content and save to `outputs/`
4. **Output Files**: 
   - `{doc_id}_extracted.json` - Raw ExtractedDocument
   - `{doc_id}_extracted.md` - Markdown representation of parsed content

The **markdown/parsed content** saved in the output folder will be used by the semantic chunking stage.

In [7]:
# Check if parsed content already exists in output folder
# Look for files in various locations:
# 1. Flat structure: outputs/{doc_id}_extracted.json or .md
# 2. Docling structure: outputs/docling/markdown/{doc_id}_*/{doc_name}.md
# 3. MinerU structure: outputs/mineru/{doc_name}.md or .json
doc_id_base = pdf_path.stem  # Use filename without extension as base

# Search for existing parsed content
existing_json = None
existing_md = None

# Check flat structure first
flat_json = OUTPUT_DIR / f"{doc_id_base}_extracted.json"
flat_md = OUTPUT_DIR / f"{doc_id_base}_extracted.md"
if flat_json.exists():
    existing_json = flat_json
elif flat_md.exists():
    existing_md = flat_md

# Check Docling structure: outputs/docling/markdown/{doc_id}_*/{doc_name}.md
if not existing_json and not existing_md:
    docling_md_dir = OUTPUT_DIR / "docling" / "markdown"
    if docling_md_dir.exists():
        # Search for directories matching the doc_id pattern
        for subdir in docling_md_dir.glob(f"{doc_id_base}*"):
            md_file = subdir / f"{doc_id_base}.md"
            if md_file.exists():
                existing_md = md_file
                print(f"📁 Found Docling markdown: {md_file}")
                break

# Check MinerU structure
if not existing_json and not existing_md:
    mineru_dir = OUTPUT_DIR / "mineru"
    if mineru_dir.exists():
        mineru_json = mineru_dir / f"{doc_id_base}.json"
        mineru_md = mineru_dir / f"{doc_id_base}.md"
        if mineru_json.exists():
            existing_json = mineru_json
        elif mineru_md.exists():
            existing_md = mineru_md

if existing_json or existing_md:
    print("📁 Found existing parsed content in output folder!")
    found_file = existing_json if existing_json else existing_md
    print(f"  → {found_file}")
    print("  → Skipping extraction - will use existing parsed content for chunking\n")
    SKIP_EXTRACTION = True  # Automatically skip if found
else:
    SKIP_EXTRACTION = False
    print("📄 No existing parsed content found. Running extraction...\n")

if not SKIP_EXTRACTION:
    # Initialize triage agent and extraction router
    triage_agent = TriageAgent(profiles_dir=PROFILES_DIR)
    extraction_router = ExtractionRouter()
    
    # Classify document
    print("🔍 Running triage analysis...")
    profile = triage_agent.classify_document(pdf_path)

    print(f"\n📊 Document Profile:")
    print(f"  Document ID: {profile.doc_id}")
    print(f"  Origin Type: {profile.origin_type}")
    print(f"  Layout Complexity: {profile.layout_complexity}")
    print(f"  Domain Hint: {profile.domain_hint}")
    print(f"  Estimated Cost: {profile.estimated_cost}")
    print(f"  Page Count: {profile.metadata.page_count}")
    
    # Extract document - This uses OCR tools (Docling/MinerU/VisionExtractor)
    # and normalizes their raw output into ExtractedDocument
    print("\n📄 Stage 2: Extracting document with OCR tools...")
    print("  → OCR Tools extract raw content (text, tables, figures)")
    print("  → Raw output is normalized to ExtractedDocument schema")
    print("  → ExtractedDocument will be saved to output folder\n")
    
    extracted_doc = extraction_router.extract(profile, str(pdf_path))
    doc_id = profile.doc_id
else:
    print("⏭️  Skipping extraction - using existing parsed content from output folder")
    # We'll load the parsed content and create extracted_doc in Step 3
    doc_id = doc_id_base
    print(f"  → Document ID: {doc_id}")
    print(f"  → Parsed content will be loaded in Step 3 (Semantic Chunking)")
    print(f"  → Proceed to Step 3 to load and chunk the existing parsed content\n")

    # Show what strategy was used (which OCR tool)
    print(f"✓ Extraction Complete (Raw OCR Output → ExtractedDocument):")
    print(f"  Text Blocks: {len(extracted_doc.text_blocks)}")
    print(f"  Tables: {len(extracted_doc.tables)}")
    print(f"  Figures: {len(extracted_doc.figures)}")
    print(f"  Reading Order: {len(extracted_doc.reading_order)} indices")
    
    # Verify ExtractedDocument schema compliance
    print("\n🔍 Verifying ExtractedDocument Schema (Normalized OCR Output):")
    print(f"  ✓ Has text_blocks: {hasattr(extracted_doc, 'text_blocks')}")
    print(f"  ✓ Has tables: {hasattr(extracted_doc, 'tables')}")
    print(f"  ✓ Has figures: {hasattr(extracted_doc, 'figures')}")
    print(f"  ✓ Has reading_order: {hasattr(extracted_doc, 'reading_order')}")
    
    # Check text blocks have bounding boxes
    if extracted_doc.text_blocks:
        blocks_with_bbox = sum(1 for b in extracted_doc.text_blocks if hasattr(b, 'bbox') and b.bbox)
        print(f"  ✓ Text blocks with bbox: {blocks_with_bbox}/{len(extracted_doc.text_blocks)}")
    
    # Check tables have headers and rows
    if extracted_doc.tables:
        tables_with_structure = sum(1 for t in extracted_doc.tables if t.headers or t.rows)
        print(f"  ✓ Tables with structure: {tables_with_structure}/{len(extracted_doc.tables)}")
    
    # Check figures have captions/bbox
    if extracted_doc.figures:
        figures_with_bbox = sum(1 for f in extracted_doc.figures if hasattr(f, 'bbox') and f.bbox)
        print(f"  ✓ Figures with bbox: {figures_with_bbox}/{len(extracted_doc.figures)}")
    
    # Save raw extracted document as JSON
    extracted_json_path = OUTPUT_DIR / f"{doc_id}_extracted.json"
    extracted_json_path.write_text(extracted_doc.model_dump_json(indent=2), encoding="utf-8")
    print(f"\n✓ Raw extracted content (ExtractedDocument JSON) saved to: {extracted_json_path}")
    
    # Convert ExtractedDocument to markdown (parsed content)
    def extracted_doc_to_markdown(doc: ExtractedDocument, title: str = "Extracted Document") -> str:
        """Convert ExtractedDocument to Markdown format."""
        lines = [f"# {title}\n"]
        lines.append(f"**Extracted from:** {pdf_path.name}\n")
        lines.append(f"**Text Blocks:** {len(doc.text_blocks)}, **Tables:** {len(doc.tables)}, **Figures:** {len(doc.figures)}\n")
        
        # Text blocks
        if doc.text_blocks:
            lines.append("## Text Blocks\n")
            for i, block in enumerate(doc.text_blocks, 1):
                lines.append(f"### Text Block {i} (Page {block.page_num})")
                lines.append(f"**Bounding Box:** ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")
                lines.append(f"\n{block.content}\n")
        
        # Tables
        if doc.tables:
            lines.append("## Tables\n")
            for i, table in enumerate(doc.tables, 1):
                lines.append(f"### Table {i} (Page {table.page_num})")
                lines.append(f"**Bounding Box:** ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")
                lines.append("\n")
                if table.headers:
                    lines.append("| " + " | ".join(table.headers) + " |")
                    lines.append("| " + " | ".join(["---"] * len(table.headers)) + " |")
                for row in table.rows:
                    lines.append("| " + " | ".join(str(cell) for cell in row) + " |")
                lines.append("\n")
        
        # Figures
        if doc.figures:
            lines.append("## Figures\n")
            for i, figure in enumerate(doc.figures, 1):
                lines.append(f"### Figure {i} (Page {figure.page_num})")
                lines.append(f"**Bounding Box:** ({figure.bbox.x0:.1f}, {figure.bbox.y0:.1f}) → ({figure.bbox.x1:.1f}, {figure.bbox.y1:.1f})")
                if figure.caption:
                    lines.append(f"**Caption:** {figure.caption}")
                lines.append("\n")
        
        return "\n".join(lines)
    
    # Save markdown (parsed content) to output folder
    extracted_md = extracted_doc_to_markdown(extracted_doc, f"Raw Extracted Content: {pdf_path.name}")
    extracted_md_path = OUTPUT_DIR / f"{doc_id}_extracted.md"
    extracted_md_path.write_text(extracted_md, encoding="utf-8")
    print(f"✓ Parsed content (Markdown) saved to: {extracted_md_path}")
    print(f"\n  → These files in {OUTPUT_DIR} will be used by semantic chunking stage:")
    print(f"    - {extracted_json_path.name} (JSON)")
    print(f"    - {extracted_md_path.name} (Markdown)")

📁 Found existing parsed content in output folder!
  → c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_extracted.json
  → Skipping extraction - will use existing parsed content for chunking

⏭️  Skipping extraction - using existing parsed content from output folder
  → Document ID: 2018_Audited_Financial_Statement_Report
  → Parsed content will be loaded in Step 3 (Semantic Chunking)
  → Proceed to Step 3 to load and chunk the existing parsed content

✓ Extraction Complete (Raw OCR Output → ExtractedDocument):
  Text Blocks: 1
  Tables: 3
  Figures: 8
  Reading Order: 1 indices

🔍 Verifying ExtractedDocument Schema (Normalized OCR Output):
  ✓ Has text_blocks: True
  ✓ Has tables: True
  ✓ Has figures: True
  ✓ Has reading_order: True
  ✓ Text blocks with bbox: 1/1
  ✓ Tables with structure: 3/3
  ✓ Figures with bbox: 8/8

✓ Raw extracted content (ExtractedDocument JSON) saved to: c:\Users\Hel

## Step 3: Semantic Chunking - Load Parsed Content from Output Folder

**Pipeline Flow:**
1. **Load from Output Folder**: Read markdown/parsed content from `.refinery/analyses/{doc_id}_extracted.md` or `{doc_id}_extracted.json`
2. **ChunkingEngine**: Processes the parsed content and applies chunking rules
3. **Output**: `List[LDU]` (RAG-ready semantic chunks)

The ChunkingEngine loads the **parsed content** (markdown/JSON) from the output folder and converts it into semantically coherent LDUs. Each LDU must have:
- `content`: str
- `chunk_type`: Literal['paragraph', 'table', 'figure', 'list', 'header', 'footnote']
- `page_refs`: List[int]
- `bounding_box`: Dict[str, float]
- `parent_section`: Optional[str]
- `token_count`: int
- `content_hash`: str

In [8]:
# Initialize chunking engine
chunking_engine = ChunkingEngine()

# Load parsed content from output folder
print("🧩 Stage 3: Running Semantic Chunking Engine...")
print("  Loading parsed content from output folder...")

# Determine doc_id - use from extraction if available, otherwise from filename
if 'doc_id' not in locals():
    doc_id = pdf_path.stem  # Use filename without extension

# Search for existing parsed content (same logic as Step 2)
found_json = None
found_md = None

# Check flat structure
flat_json = OUTPUT_DIR / f"{doc_id}_extracted.json"
flat_md = OUTPUT_DIR / f"{doc_id}_extracted.md"
if flat_json.exists():
    found_json = flat_json
elif flat_md.exists():
    found_md = flat_md

# Check Docling structure
if not found_json and not found_md:
    docling_md_dir = OUTPUT_DIR / "docling" / "markdown"
    if docling_md_dir.exists():
        for subdir in docling_md_dir.glob(f"{doc_id}*"):
            md_file = subdir / f"{doc_id}.md"
            if md_file.exists():
                found_md = md_file
                break

# Check MinerU structure
if not found_json and not found_md:
    mineru_dir = OUTPUT_DIR / "mineru"
    if mineru_dir.exists():
        mineru_json = mineru_dir / f"{doc_id}.json"
        mineru_md = mineru_dir / f"{doc_id}.md"
        if mineru_json.exists():
            found_json = mineru_json
        elif mineru_md.exists():
            found_md = mineru_md

# Load from found file
if found_json:
    print(f"  → Loading from: {found_json}")
    # Load ExtractedDocument from JSON
    extracted_doc_data = json.loads(found_json.read_text(encoding="utf-8"))
    extracted_doc = ExtractedDocument(**extracted_doc_data)
    print(f"  ✓ Loaded ExtractedDocument from JSON")
    print(f"    Text Blocks: {len(extracted_doc.text_blocks)}")
    print(f"    Tables: {len(extracted_doc.tables)}")
    print(f"    Figures: {len(extracted_doc.figures)}")
elif found_md:
    print(f"  → Found markdown file: {found_md}")
    print(f"  → Converting markdown to ExtractedDocument...")
    
    # Parse markdown to create ExtractedDocument
    md_content = found_md.read_text(encoding="utf-8")
    
    from src.models.extracted_document import TextBlock, Table, Figure, BoundingBox
    
    text_blocks = []
    tables = []
    figures = []
    page_num = 1
    
    lines = md_content.split('\n')
    current_text_block = []
    current_table_rows = []
    in_table = False
    table_headers = None
    
    for i, line in enumerate(lines):
        stripped = line.strip()
        
        # Detect markdown tables (starts with | and contains |)
        if '|' in stripped and stripped.startswith('|'):
            # Skip table separator rows (|---|---|)
            if all(c in ['-', '|', ':', ' '] for c in stripped):
                continue
            
            if not in_table:
                in_table = True
                # Extract headers from first table row
                if table_headers is None:
                    headers = [cell.strip() for cell in stripped.split('|')[1:-1]]
                    table_headers = headers
                    current_table_rows = []
            else:
                # Data row
                cells = [cell.strip() for cell in stripped.split('|')[1:-1]]
                if len(cells) == len(table_headers) if table_headers else True:
                    current_table_rows.append(cells)
            continue
        elif in_table:
            # End of table - save it
            if current_table_rows and table_headers:
                tables.append(Table(
                    headers=table_headers,
                    rows=current_table_rows,
                    bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                    page_num=page_num
                ))
            in_table = False
            table_headers = None
            current_table_rows = []
        
        # Detect images/figures
        if stripped == '<!-- image -->':
            figures.append(Figure(
                caption="",
                bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                page_num=page_num
            ))
            continue
        
        # Regular text content
        if stripped and not stripped.startswith('#'):
            current_text_block.append(line)
        elif stripped.startswith('#'):
            # Header - save current block and start new one
            if current_text_block:
                text = '\n'.join(current_text_block).strip()
                if text:
                    text_blocks.append(TextBlock(
                        content=text,
                        bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                        page_num=page_num
                    ))
                current_text_block = []
    
    # Save last text block
    if current_text_block:
        text = '\n'.join(current_text_block).strip()
        if text:
            text_blocks.append(TextBlock(
                content=text,
                bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
                page_num=page_num
            ))
    
    # Save last table if still in table
    if in_table and current_table_rows:
        tables.append(Table(
            headers=table_headers or [],
            rows=current_table_rows,
            bbox=BoundingBox(x0=0.0, y0=0.0, x1=612.0, y1=792.0),
            page_num=page_num
        ))
    
    # Create ExtractedDocument from parsed markdown
    extracted_doc = ExtractedDocument(
        text_blocks=text_blocks,
        tables=tables,
        figures=figures,
        reading_order=list(range(len(text_blocks)))
    )
    print(f"  ✓ Created ExtractedDocument from markdown")
    print(f"    Text Blocks: {len(extracted_doc.text_blocks)}")
    print(f"    Tables: {len(extracted_doc.tables)}")
    print(f"    Figures: {len(extracted_doc.figures)}")
elif 'extracted_doc' in locals():
    print(f"  → Using in-memory ExtractedDocument from Step 2")
else:
    raise FileNotFoundError(
        f"No parsed content found in {OUTPUT_DIR}\n"
        f"Searched locations:\n"
        f"  - {OUTPUT_DIR}/{doc_id}_extracted.json\n"
        f"  - {OUTPUT_DIR}/{doc_id}_extracted.md\n"
        f"  - {OUTPUT_DIR}/docling/markdown/{doc_id}*/\n"
        f"  - {OUTPUT_DIR}/mineru/{doc_id}.*\n"
        f"Please ensure parsed content exists in one of these locations."
    )

print(f"\n  Input: Parsed content from output folder")
print(f"  Processing: Applying chunking rules to parsed content")
print(f"  Output: List[LDU] (RAG-ready semantic chunks)\n")

# Pass the parsed content to chunking engine
chunks = chunking_engine.chunk(extracted_doc)

print(f"✓ Chunking Complete:")
print(f"  Total LDUs Generated: {len(chunks)}")

# Verify LDU structure - check all required fields
print("\n🔍 Verifying LDU Structure (Required Fields):")
required_fields = ['content', 'chunk_type', 'page_refs', 'bounding_box', 'parent_section', 'token_count', 'content_hash']
missing_fields = []

for i, chunk in enumerate(chunks[:5]):  # Check first 5 chunks
    chunk_missing = []
    for field in required_fields:
        if not hasattr(chunk, field):
            chunk_missing.append(field)
        elif field == 'content_hash' and not chunk.content_hash:
            chunk_missing.append(f'{field} (empty)')
    if chunk_missing:
        missing_fields.append((i, chunk_missing))

if missing_fields:
    print(f"  ⚠️  Found chunks with missing fields:")
    for idx, fields in missing_fields:
        print(f"    Chunk {idx}: missing {', '.join(fields)}")
else:
    print(f"  ✓ All chunks have required fields")

# Verify chunk types are valid
valid_chunk_types = ['paragraph', 'table', 'figure', 'list', 'header', 'footnote']
invalid_types = [c for c in chunks if c.chunk_type not in valid_chunk_types]
if invalid_types:
    print(f"  ⚠️  Found {len(invalid_types)} chunks with invalid chunk_type")
else:
    print(f"  ✓ All chunk types are valid")

# Analyze chunk distribution
chunk_types = {}
for chunk in chunks:
    chunk_types[chunk.chunk_type] = chunk_types.get(chunk.chunk_type, 0) + 1

print("\n📊 Chunk Type Distribution:")
for chunk_type, count in sorted(chunk_types.items()):
    print(f"  {chunk_type}: {count}")

# Token statistics
total_tokens = sum(c.token_count for c in chunks)
avg_tokens = total_tokens / len(chunks) if chunks else 0
max_tokens = max((c.token_count for c in chunks), default=0)
min_tokens = min((c.token_count for c in chunks), default=0)

print(f"\n📊 Token Statistics:")
print(f"  Total Tokens: {total_tokens:,}")
print(f"  Average per Chunk: {avg_tokens:.1f}")
print(f"  Min: {min_tokens}, Max: {max_tokens}")

# Save chunks
chunks_path = OUTPUT_DIR / f"{doc_id}_chunks.json"
chunks_data = [chunk.model_dump() for chunk in chunks]
chunks_path.write_text(json.dumps(chunks_data, indent=2), encoding="utf-8")
print(f"\n✓ Chunks saved to: {chunks_path}")

🧩 Stage 3: Running Semantic Chunking Engine...
  Loading parsed content from output folder...
  → Loading from: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_extracted.json
  ✓ Loaded ExtractedDocument from JSON
    Text Blocks: 1
    Tables: 3
    Figures: 8

  Input: Parsed content from output folder
  Processing: Applying chunking rules to parsed content
  Output: List[LDU] (RAG-ready semantic chunks)

✓ Chunking Complete:
  Total LDUs Generated: 14

🔍 Verifying LDU Structure (Required Fields):
  ✓ All chunks have required fields
  ✓ All chunk types are valid

📊 Chunk Type Distribution:
  figure: 8
  paragraph: 1
  table: 5

📊 Token Statistics:
  Total Tokens: 2,420
  Average per Chunk: 172.9
  Min: 3, Max: 579

✓ Chunks saved to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_chunks.json


## Step 4: Test Chunking Rules

Test all 5 core chunking rules to ensure data quality.

In [9]:
# Run comprehensive validation
print("🔍 Validating Chunking Rules...\n")
try:
    overall_success, results, violations = chunking_engine.validate_chunks_detailed(chunks)
except Exception as e:
    print(f"⚠️  Validation error: {e}")
    # Fallback to simple validation
    overall_success = chunking_engine.validate_chunks(chunks)
    results = {"overall": overall_success}
    violations = []

# Test each rule individually
print("Rule Validation Results:")
print("=" * 60)

# Initialize rule variables with defaults
rule1_passed = results.get("rule1_table_integrity", False) if results else False
rule1_violations = [v for v in violations if v.get("rule") == "rule1_table_integrity"] if violations else []
print(f"\n📋 Rule 1: Table cell never split from header row")
print(f"  Status: {'✓ PASS' if rule1_passed else '✗ FAIL'}")
if rule1_violations:
    print(f"  Violations: {len(rule1_violations)}")
    for v in rule1_violations[:3]:
        print(f"    - {v.get('violation', 'Unknown')} (chunk: {v.get('chunk_id', 'N/A')[:16]}...)")
else:
    print(f"  ✓ All table chunks preserve header integrity")

# Rule 2: Figure caption stored as metadata
rule2_passed = results.get("rule2_figure_captions", False) if results else False
rule2_violations = [v for v in violations if v.get("rule") == "rule2_figure_captions"] if violations else []
print(f"\n🖼️  Rule 2: Figure caption stored as metadata")
print(f"  Status: {'✓ PASS' if rule2_passed else '✗ FAIL'}")
if rule2_violations:
    print(f"  Violations: {len(rule2_violations)}")
    for v in rule2_violations[:3]:
        print(f"    - {v.get('violation', 'Unknown')} (chunk: {v.get('chunk_id', 'N/A')[:16]}...)")
else:
    figure_chunks = [c for c in chunks if c.chunk_type == "figure"]
    captions_in_metadata = sum(1 for c in figure_chunks if "caption" in c.metadata)
    print(f"  ✓ {captions_in_metadata}/{len(figure_chunks)} figure chunks have caption metadata")

# Rule 3: Numbered list kept as single LDU
rule3_passed = results.get("rule3_list_preservation", False) if results else False
rule3_violations = [v for v in violations if v.get("rule") == "rule3_list_preservation"] if violations else []
print(f"\n📝 Rule 3: Numbered list kept as single LDU")
print(f"  Status: {'✓ PASS' if rule3_passed else '✗ FAIL'}")
if rule3_violations:
    print(f"  Violations: {len(rule3_violations)}")
    for v in rule3_violations[:3]:
        print(f"    - {v.get('violation', 'Unknown')} (chunk: {v.get('chunk_id', 'N/A')[:16]}...)")
else:
    list_chunks = [c for c in chunks if c.chunk_type == "list"]
    print(f"  ✓ {len(list_chunks)} list chunks preserved as single LDUs")

# Rule 4: Section headers as parent metadata
rule4_passed = results.get("rule4_section_hierarchy", False) if results else False
rule4_violations = [v for v in violations if v.get("rule") == "rule4_section_hierarchy"] if violations else []
print(f"\n📑 Rule 4: Section headers as parent metadata")
print(f"  Status: {'✓ PASS' if rule4_passed else '✗ FAIL'}")
if rule4_violations:
    print(f"  Violations: {len(rule4_violations)}")
    for v in rule4_violations[:3]:
        print(f"    - {v.get('violation', 'Unknown')} (chunk: {v.get('chunk_id', 'N/A')[:16]}...)")
else:
    chunks_with_section = sum(1 for c in chunks if c.parent_section)
    print(f"  ✓ {chunks_with_section}/{len(chunks)} chunks have parent_section assigned")

# Rule 5: Cross-references resolved
rule5_passed = results.get("rule5_cross_references", False) if results else False
rule5_violations = [v for v in violations if v.get("rule") == "rule5_cross_references"] if violations else []
print(f"\n🔗 Rule 5: Cross-references resolved")
print(f"  Status: {'✓ PASS' if rule5_passed else '✗ FAIL'}")
if rule5_violations:
    print(f"  Violations: {len(rule5_violations)}")
    for v in rule5_violations[:3]:
        print(f"    - {v.get('violation', 'Unknown')} (target: {v.get('target_id', 'N/A')[:16]}...)")
else:
    chunks_with_refs = sum(1 for c in chunks if c.cross_references)
    total_refs = sum(len(c.cross_references) for c in chunks)
    print(f"  ✓ {chunks_with_refs} chunks have {total_refs} cross-references resolved")

print("\n" + "=" * 60)
print(f"\nOverall Validation: {'✓ PASSED' if overall_success else '✗ FAILED'}")
print(f"Total Violations: {len(violations)}")

# Save validation results
validation_results = {
    "overall_success": overall_success,
    "rule_results": results,
    "violations": violations,
    "violation_summary": {
        "rule1_table_integrity": len(rule1_violations),
        "rule2_figure_captions": len(rule2_violations),
        "rule3_list_preservation": len(rule3_violations),
        "rule4_section_hierarchy": len(rule4_violations),
        "rule5_cross_references": len(rule5_violations),
    },
}

validation_path = OUTPUT_DIR / f"{doc_id}_validation.json"
validation_path.write_text(json.dumps(validation_results, indent=2), encoding="utf-8")
print(f"\n✓ Validation results saved to: {validation_path}")

🔍 Validating Chunking Rules...

Rule Validation Results:

📋 Rule 1: Table cell never split from header row
  Status: ✓ PASS
  ✓ All table chunks preserve header integrity

🖼️  Rule 2: Figure caption stored as metadata
  Status: ✓ PASS
  ✓ 8/8 figure chunks have caption metadata

📝 Rule 3: Numbered list kept as single LDU
  Status: ✓ PASS
  ✓ 0 list chunks preserved as single LDUs

📑 Rule 4: Section headers as parent metadata
  Status: ✓ PASS
  ✓ 14/14 chunks have parent_section assigned

🔗 Rule 5: Cross-references resolved
  Status: ✓ PASS
  ✓ 0 chunks have 0 cross-references resolved


Overall Validation: ✓ PASSED
Total Violations: 0

✓ Validation results saved to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_validation.json


## Step 5: Detailed Rule Testing

Examine specific examples of each rule in action.

In [10]:
# Rule 1: Table Integrity Examples
table_chunks = [c for c in chunks if c.chunk_type == "table"]
if table_chunks:
    print("📋 Rule 1 Examples: Table Integrity")
    print("=" * 60)
    for i, chunk in enumerate(table_chunks[:2], 1):  # Show first 2 table chunks
        print(f"\nTable Chunk {i}:")
        print(f"  Hash: {chunk.content_hash[:16]}...")
        print(f"  Pages: {chunk.page_refs}")
        print(f"  Tokens: {chunk.token_count}")
        has_headers = "table_headers" in chunk.metadata
        print(f"  Has Headers in Metadata: {'✓' if has_headers else '✗'}")
        if has_headers:
            headers = chunk.metadata.get("table_headers", [])
            print(f"  Headers: {headers[:5]}..." if len(headers) > 5 else f"  Headers: {headers}")
        print(f"  Content Preview: {chunk.content[:100]}...")
else:
    print("📋 Rule 1: No table chunks found in document")

# Rule 2: Figure Caption Examples
figure_chunks = [c for c in chunks if c.chunk_type == "figure"]
if figure_chunks:
    print("\n\n🖼️  Rule 2 Examples: Figure Captions")
    print("=" * 60)
    for i, chunk in enumerate(figure_chunks[:2], 1):
        print(f"\nFigure Chunk {i}:")
        print(f"  Hash: {chunk.content_hash[:16]}...")
        print(f"  Pages: {chunk.page_refs}")
        has_caption = "caption" in chunk.metadata
        print(f"  Has Caption in Metadata: {'✓' if has_caption else '✗'}")
        if has_caption:
            caption = chunk.metadata.get("caption", "")
            print(f"  Caption: {caption[:100]}..." if len(caption) > 100 else f"  Caption: {caption}")
        print(f"  Content: {chunk.content[:100]}...")
else:
    print("\n\n🖼️  Rule 2: No figure chunks found in document")

# Rule 3: List Preservation Examples
list_chunks = [c for c in chunks if c.chunk_type == "list"]
if list_chunks:
    print("\n\n📝 Rule 3 Examples: List Preservation")
    print("=" * 60)
    for i, chunk in enumerate(list_chunks[:2], 1):
        print(f"\nList Chunk {i}:")
        print(f"  Hash: {chunk.content_hash[:16]}...")
        print(f"  Pages: {chunk.page_refs}")
        print(f"  Tokens: {chunk.token_count}")
        is_list = chunk.metadata.get("is_list", False)
        list_type = chunk.metadata.get("list_type", "unknown")
        item_count = chunk.metadata.get("item_count", 0)
        print(f"  Is List: {'✓' if is_list else '✗'}")
        print(f"  List Type: {list_type}")
        print(f"  Item Count: {item_count}")
        print(f"  Content Preview: {chunk.content[:150]}...")
else:
    print("\n\n📝 Rule 3: No list chunks found in document")

# Rule 4: Section Hierarchy Examples
chunks_with_section = [c for c in chunks if c.parent_section]
if chunks_with_section:
    print("\n\n📑 Rule 4 Examples: Section Hierarchy")
    print("=" * 60)
    # Group by section
    sections = {}
    for chunk in chunks_with_section:
        section = chunk.parent_section
        if section not in sections:
            sections[section] = []
        sections[section].append(chunk)
    
    for section_name, section_chunks in list(sections.items())[:3]:  # Show first 3 sections
        print(f"\nSection: {section_name}")
        print(f"  Chunks in Section: {len(section_chunks)}")
        print(f"  Chunk Types: {', '.join(set(c.chunk_type for c in section_chunks))}")
        # Show first chunk in section
        if section_chunks:
            first = section_chunks[0]
            print(f"  Example Chunk:")
            print(f"    Type: {first.chunk_type}")
            print(f"    Pages: {first.page_refs}")
            print(f"    Content Preview: {first.content[:100]}...")
else:
    print("\n\n📑 Rule 4: No chunks with parent_section found")

# Rule 5: Cross-Reference Examples
chunks_with_refs = [c for c in chunks if c.cross_references]
if chunks_with_refs:
    print("\n\n🔗 Rule 5 Examples: Cross-References")
    print("=" * 60)
    for i, chunk in enumerate(chunks_with_refs[:3], 1):
        print(f"\nChunk with References {i}:")
        print(f"  Hash: {chunk.content_hash[:16]}...")
        print(f"  Type: {chunk.chunk_type}")
        print(f"  Cross-References: {len(chunk.cross_references)}")
        for ref in chunk.cross_references:
            print(f"    - {ref.reference_type}: \"{ref.anchor_text}\" → {ref.target_id[:16]}...")
        print(f"  Content Preview: {chunk.content[:100]}...")
else:
    print("\n\n🔗 Rule 5: No chunks with cross-references found")

📋 Rule 1 Examples: Table Integrity

Table Chunk 1:
  Hash: 6e2f239aca4302d2...
  Pages: [1]
  Tokens: 579
  Has Headers in Metadata: ✓
  Headers: ['', 'Notes', "30June2018 Birr'ooo", "30June2017 Birr'ooo"]
  Content Preview:  | Notes | 30June2018 Birr'ooo | 30June2017 Birr'ooo
Revenue from contractswithcustomers | 5 | 34,81...

Table Chunk 2:
  Hash: b473541d2cbdd3e6...
  Pages: [1]
  Tokens: 467
  Has Headers in Metadata: ✓
  Headers: ['ASSETS', 'Notes', "30June2018 Birr'ooo", "30June2017 Birr'ooo", "1July2016 Birr'ooo"]
  Content Preview: ASSETS | Notes | 30June2018 Birr'ooo | 30June2017 Birr'ooo | 1July2016 Birr'ooo
Noncurrentassets |  ...


🖼️  Rule 2 Examples: Figure Captions

Figure Chunk 1:
  Hash: 02faec9381d86985...
  Pages: [1]
  Has Caption in Metadata: ✓
  Caption: 
  Content: [Figure]...

Figure Chunk 2:
  Hash: 02faec9381d86985...
  Pages: [1]
  Has Caption in Metadata: ✓
  Caption: 
  Content: [Figure]...


📝 Rule 3: No list chunks found in document


📑 Rule 4 Examples: 

## Step 6: Convert Chunks to Markdown

Convert the resulting LDUs to Markdown format for easy inspection. Note that the parsed content (ExtractedDocument) was already saved as markdown in Step 2. This step exports:
- Semantic chunks (LDUs) created from the parsed content in the output folder
- Rule verification summary showing how chunking rules were applied

In [11]:
def chunks_to_markdown(chunks: List[LDU], title: str = "Semantic Chunks") -> str:
    """Convert LDUs to Markdown format with rule verification."""
    lines = [f"# {title}\n"]
    lines.append(f"**Total Chunks:** {len(chunks)}\n")
    lines.append("## Chunking Rules Verification\n")
    
    # Rule verification summary
    table_chunks = [c for c in chunks if c.chunk_type == "table"]
    figure_chunks = [c for c in chunks if c.chunk_type == "figure"]
    list_chunks = [c for c in chunks if c.chunk_type == "list"]
    chunks_with_section = [c for c in chunks if c.parent_section]
    chunks_with_refs = [c for c in chunks if c.cross_references]
    
    lines.append(f"- **Rule 1 (Table Integrity):** {len(table_chunks)} table chunks")
    lines.append(f"- **Rule 2 (Figure Captions):** {len(figure_chunks)} figure chunks")
    lines.append(f"- **Rule 3 (List Preservation):** {len(list_chunks)} list chunks")
    lines.append(f"- **Rule 4 (Section Hierarchy):** {len(chunks_with_section)} chunks with parent_section")
    lines.append(f"- **Rule 5 (Cross-References):** {sum(len(c.cross_references) for c in chunks)} cross-references\n")
    
    # Group by section
    sections = {}
    no_section = []
    
    for chunk in chunks:
        section = chunk.parent_section or "No Section"
        if section == "No Section":
            no_section.append(chunk)
        else:
            if section not in sections:
                sections[section] = []
            sections[section].append(chunk)
    
    # Write sections
    for section_name, section_chunks in sorted(sections.items()):
        lines.append(f"## {section_name}\n")
        lines.append(f"*{len(section_chunks)} chunks in this section*\n")
        
        for i, chunk in enumerate(section_chunks, 1):
            lines.append(f"### Chunk {i}: {chunk.chunk_type.upper()}")
            lines.append(f"- **Hash:** `{chunk.content_hash[:16]}...`")
            lines.append(f"- **Pages:** {', '.join(map(str, chunk.page_refs))}")
            lines.append(f"- **Tokens:** {chunk.token_count}")
            if chunk.bounding_box:
                bbox = chunk.bounding_box
                lines.append(f"- **BBox:** ({bbox.get('x0', 0):.1f}, {bbox.get('y0', 0):.1f}) → ({bbox.get('x1', 0):.1f}, {bbox.get('y1', 0):.1f})")
            if chunk.metadata:
                # Highlight rule-specific metadata
                if chunk.chunk_type == "table" and "table_headers" in chunk.metadata:
                    lines.append(f"- **Rule 1 ✓:** Headers preserved: {chunk.metadata['table_headers'][:3]}...")
                if chunk.chunk_type == "figure" and "caption" in chunk.metadata:
                    lines.append(f"- **Rule 2 ✓:** Caption in metadata: {chunk.metadata['caption'][:50]}...")
                if chunk.chunk_type == "list" and "is_list" in chunk.metadata:
                    lines.append(f"- **Rule 3 ✓:** List preserved ({chunk.metadata.get('item_count', 0)} items)")
                if chunk.cross_references:
                    lines.append(f"- **Rule 5 ✓:** {len(chunk.cross_references)} cross-reference(s)")
            lines.append(f"\n```\n{chunk.content[:500]}{'...' if len(chunk.content) > 500 else ''}\n```\n")
    
    # Write chunks without section
    if no_section:
        lines.append("## No Section\n")
        lines.append(f"*{len(no_section)} chunks without section assignment*\n")
        for i, chunk in enumerate(no_section, 1):
            lines.append(f"### Chunk {i}: {chunk.chunk_type.upper()}")
            lines.append(f"- **Hash:** `{chunk.content_hash[:16]}...`")
            lines.append(f"- **Pages:** {', '.join(map(str, chunk.page_refs))}")
            lines.append(f"- **Tokens:** {chunk.token_count}")
            lines.append(f"\n```\n{chunk.content[:500]}{'...' if len(chunk.content) > 500 else ''}\n```\n")
    
    return "\n".join(lines)


# Convert chunks to markdown (parsed content was already saved in Step 2)
print("📝 Converting Chunks to Markdown...")
if found_md:
    print(f"  → Parsed content source: {found_md.name}")
elif found_json:
    print(f"  → Parsed content source: {found_json.name}")
print("  → Exporting semantic chunks (LDUs) from chunking engine\n")

# Chunks markdown (semantic chunks from parsed content)
chunks_md = chunks_to_markdown(chunks, f"Semantic Chunks: {pdf_path.name}")
chunks_md_path = OUTPUT_DIR / f"{doc_id}_chunks.md"
chunks_md_path.write_text(chunks_md, encoding="utf-8")
print(f"✓ Semantic chunks (LDUs) markdown saved to: {chunks_md_path}")
print(f"  → These are the RAG-ready chunks created from parsed content in output folder")

print(f"\n📊 Final Summary:")
print(f"  Document: {pdf_path.name}")
print(f"  Extracted: {len(extracted_doc.text_blocks)} blocks, {len(extracted_doc.tables)} tables, {len(extracted_doc.figures)} figures")
print(f"  Chunked: {len(chunks)} LDUs ({total_tokens:,} tokens)")
print(f"  Validation: {'✓ PASSED' if overall_success else '✗ FAILED'} ({len(violations)} violations)")
print(f"\n📁 All outputs saved to: {OUTPUT_DIR}")

📝 Converting Chunks to Markdown...
  → Parsed content source: 2018_Audited_Financial_Statement_Report_extracted.json
  → Exporting semantic chunks (LDUs) from chunking engine

✓ Semantic chunks (LDUs) markdown saved to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs\2018_Audited_Financial_Statement_Report_chunks.md
  → These are the RAG-ready chunks created from parsed content in output folder

📊 Final Summary:
  Document: 2018_Audited_Financial_Statement_Report.pdf
  Extracted: 1 blocks, 3 tables, 8 figures
  Chunked: 14 LDUs (2,420 tokens)
  Validation: ✓ PASSED (0 violations)

📁 All outputs saved to: c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\outputs


## Step 7: Summary Report

Generate a comprehensive summary of the semantic chunking test results.

In [12]:
# Create comprehensive analysis summary
analysis_summary = {
    "document": {
        "path": str(pdf_path),
        "name": pdf_path.name,
        "size_mb": pdf_path.stat().st_size / 1024 / 1024,
    },
    "profile": {
        "doc_id": profile.doc_id,
        "origin_type": profile.origin_type,
        "layout_complexity": profile.layout_complexity,
        "domain_hint": profile.domain_hint,
        "estimated_cost": profile.estimated_cost,
        "page_count": profile.metadata.page_count,
    },
    "extracted_document": {
        "text_blocks": len(extracted_doc.text_blocks),
        "tables": len(extracted_doc.tables),
        "figures": len(extracted_doc.figures),
        "reading_order_length": len(extracted_doc.reading_order),
        "schema_compliant": True,  # Verified in Step 2
    },
    "chunking": {
        "total_chunks": len(chunks),
        "chunk_types": chunk_types,
        "token_stats": {
            "total": total_tokens,
            "average": avg_tokens,
            "min": min_tokens,
            "max": max_tokens,
        },
        "ldu_structure_verified": {
            "all_have_content": all(hasattr(c, 'content') and c.content for c in chunks),
            "all_have_chunk_type": all(hasattr(c, 'chunk_type') and c.chunk_type in valid_chunk_types for c in chunks),
            "all_have_page_refs": all(hasattr(c, 'page_refs') and c.page_refs for c in chunks),
            "all_have_bounding_box": all(hasattr(c, 'bounding_box') for c in chunks),
            "all_have_token_count": all(hasattr(c, 'token_count') and c.token_count >= 0 for c in chunks),
            "all_have_content_hash": all(hasattr(c, 'content_hash') and c.content_hash for c in chunks),
        },
    },
    "chunking_rules": {
        "rule1_table_integrity": {
            "passed": rule1_passed,
            "table_chunks": len(table_chunks),
            "violations": len(rule1_violations),
        },
        "rule2_figure_captions": {
            "passed": rule2_passed,
            "figure_chunks": len(figure_chunks),
            "captions_in_metadata": sum(1 for c in figure_chunks if "caption" in c.metadata),
            "violations": len(rule2_violations),
        },
        "rule3_list_preservation": {
            "passed": rule3_passed,
            "list_chunks": len(list_chunks),
            "violations": len(rule3_violations),
        },
        "rule4_section_hierarchy": {
            "passed": rule4_passed,
            "chunks_with_section": len(chunks_with_section),
            "total_sections": len(sections),
            "violations": len(rule4_violations),
        },
        "rule5_cross_references": {
            "passed": rule5_passed,
            "chunks_with_refs": len(chunks_with_refs),
            "total_references": sum(len(c.cross_references) for c in chunks),
            "violations": len(rule5_violations),
        },
    },
    "validation": {
        "overall_success": overall_success,
        "results": results,
        "violation_count": len(violations),
        "violations": violations[:10],  # First 10 violations
    },
    "output_files": {
        "extracted_json": str(extracted_path),
        "extracted_markdown": str(extracted_md_path),
        "chunks_json": str(chunks_path),
        "chunks_markdown": str(chunks_md_path),
        "validation": str(validation_path),
    },
}

# Save analysis summary
summary_path = OUTPUT_DIR / f"{profile.doc_id}_chunking_analysis.json"
summary_path.write_text(json.dumps(analysis_summary, indent=2), encoding="utf-8")

print("📊 Semantic Chunking Engine Test Summary")
print("=" * 60)
print(f"\nDocument: {pdf_path.name}")
print(f"Pages: {profile.metadata.page_count}")
print(f"\nExtractedDocument Schema:")
print(f"  ✓ Text Blocks: {len(extracted_doc.text_blocks)}")
print(f"  ✓ Tables: {len(extracted_doc.tables)}")
print(f"  ✓ Figures: {len(extracted_doc.figures)}")
print(f"\nLDU Generation:")
print(f"  ✓ Total LDUs: {len(chunks)}")
print(f"  ✓ Total Tokens: {total_tokens:,}")
print(f"\nChunking Rules Status:")
print(f"  Rule 1 (Table Integrity): {'✓ PASS' if rule1_passed else '✗ FAIL'} ({len(table_chunks)} tables)")
print(f"  Rule 2 (Figure Captions): {'✓ PASS' if rule2_passed else '✗ FAIL'} ({len(figure_chunks)} figures)")
print(f"  Rule 3 (List Preservation): {'✓ PASS' if rule3_passed else '✗ FAIL'} ({len(list_chunks)} lists)")
print(f"  Rule 4 (Section Hierarchy): {'✓ PASS' if rule4_passed else '✗ FAIL'} ({len(chunks_with_section)} chunks with sections)")
print(f"  Rule 5 (Cross-References): {'✓ PASS' if rule5_passed else '✗ FAIL'} ({sum(len(c.cross_references) for c in chunks)} refs)")
print(f"\nOverall Validation: {'✓ PASSED' if overall_success else '✗ FAILED'} ({len(violations)} violations)")
print(f"\n✓ Complete analysis saved to: {summary_path}")
print(f"\n📁 All output files:")
for key, path in analysis_summary["output_files"].items():
    print(f"  - {key}: {Path(path).name}")

NameError: name 'profile' is not defined

In [13]:
# Create comprehensive analysis summary
analysis_summary = {
    "document": {
        "path": str(pdf_path),
        "name": pdf_path.name,
        "size_mb": pdf_path.stat().st_size / 1024 / 1024,
    },
    "profile": {
        "doc_id": profile.doc_id,
        "origin_type": profile.origin_type,
        "layout_complexity": profile.layout_complexity,
        "domain_hint": profile.domain_hint,
        "estimated_cost": profile.estimated_cost,
        "page_count": profile.metadata.page_count,
    },
    "confidence_scores": confidence_data,
    "cost_estimation": cost_comparison,
    "extraction": {
        "text_blocks": len(extracted_doc.text_blocks),
        "tables": len(extracted_doc.tables),
        "figures": len(extracted_doc.figures),
    },
    "chunking": {
        "total_chunks": len(chunks),
        "chunk_types": chunk_types,
        "token_stats": {
            "total": total_tokens,
            "average": avg_tokens,
            "min": min_tokens,
            "max": max_tokens,
        },
        "validation": {
            "overall_success": overall_success,
            "results": results,
            "violation_count": len(violations),
        },
    },
    "output_files": {
        "profile": str(profile_path),
        "confidence": str(confidence_path),
        "cost": str(cost_path),
        "extracted_json": str(extracted_path),
        "extracted_markdown": str(extracted_md_path),
        "chunks_json": str(chunks_path),
        "chunks_markdown": str(chunks_md_path),
    },
}

# Save analysis summary
summary_path = OUTPUT_DIR / f"{profile.doc_id}_analysis.json"
summary_path.write_text(json.dumps(analysis_summary, indent=2), encoding="utf-8")

print("📊 Analysis Summary:")
print(f"  Document: {pdf_path.name}")
print(f"  Recommended Strategy: {recommended}")
print(f"  Confidence: {strategies[recommended]:.3f}")
print(f"  Estimated Cost: ${cost_comparison['strategies'][recommended].get('total_cost', 0):.4f}")
print(f"  Extracted: {len(extracted_doc.text_blocks)} blocks, {len(extracted_doc.tables)} tables, {len(extracted_doc.figures)} figures")
print(f"  Chunked: {len(chunks)} LDUs ({total_tokens:,} tokens)")
print(f"  Validation: {'✓ PASSED' if overall_success else '✗ FAILED'} ({len(violations)} violations)")
print(f"\n✓ Complete analysis saved to: {summary_path}")
print(f"\n📁 All output files:")
for key, path in analysis_summary["output_files"].items():
    print(f"  - {key}: {Path(path).name}")

NameError: name 'profile' is not defined